# Multi-Output Trees & Vector-Leaf Representation

Traditional decision trees and boosting frameworks predict a single scalar target $y \in \mathbb{R}$. When tasked with predicting multiple targets $\mathbf{y} = [y_1, y_2, \dots, y_K]^T$, the standard naive approach fits $K$ completely independent models (one per target).

### Why Naive Independent Models Fail in Enterprise Systems:
1. **Ignores Target Covariance**: In quantitative finance and credit risk, outcomes are fundamentally coupled. For example, **Loss Given Default (LGD)** and **Exposure at Default (EAD)** share correlated macroeconomic sensitivities.
2. **Computational Redundancy**: Training $K$ separate trees requires $K$ separate dataset scans and histogram builds.
3. **Inference Latency**: $K$ separate models must be loaded and traversed at inference time.

### Modern XGBoost Solution: Vector-Leaf Multi-Target Trees
Modern XGBoost natively supports multi-target regression and multi-label classification. Instead of a scalar leaf weight $w_j$, each leaf in the tree stores a **$K$-dimensional weight vector** $\mathbf{w}_j \in \mathbb{R}^K$. Splitting criteria evaluate multi-dimensional gradient $\mathbf{g}_i$ and hessian $\mathbf{h}_i$ tensors to find splits that jointly optimize all targets simultaneously.

In [ ]:
import xgboost as xgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Mathematical Formulation: Multi-Target Vector Trees

Given $K$ targets, for each sample $i$, the objective function is generalized to:

$$\mathcal{L}^{(t)} = \sum_{i=1}^n l\left(\mathbf{y}_i, \hat{\mathbf{y}}_i^{(t-1)} + \mathbf{f}_t(\mathbf{x}_i)\right) + \sum_{k=1}^K \Omega(f_{t, k})$$

Using second-order Taylor expansion across each target dimension $k \in \{1, \dots, K\}$:

- First-order gradient vector: $\mathbf{g}_i = \nabla_{\hat{\mathbf{y}}} l(\mathbf{y}_i, \hat{\mathbf{y}}_i) = [g_{i,1}, g_{i,2}, \dots, g_{i,K}]^T$
- Second-order hessian vector (or diagonal): $\mathbf{h}_i = [h_{i,1}, h_{i,2}, \dots, h_{i,K}]^T$

For leaf $j$ containing sample set $I_j$, the optimal vector weight $\mathbf{w}_j^* = [w_{j,1}^*, \dots, w_{j,K}^*]^T$ is solved for each coordinate:

$$w_{j,k}^* = -\frac{\sum_{i \in I_j} g_{i,k}}{\sum_{i \in I_j} h_{i,k} + \lambda}$$

The total split gain across all $K$ targets is:

$$\text{Gain} = \frac{1}{2} \sum_{k=1}^K \left[ \frac{(\sum_{i \in I_L} g_{i,k})^2}{\sum_{i \in I_L} h_{i,k} + \lambda} + \frac{(\sum_{i \in I_R} g_{i,k})^2}{\sum_{i \in I_R} h_{i,k} + \lambda} - \frac{(\sum_{i \in I} g_{i,k})^2}{\sum_{i \in I} h_{i,k} + \lambda} \right] - \gamma$$

## 2. Financial Simulation: Joint Credit Risk (LGD & EAD)

In Basel II/III banking regulations, calculating **Expected Loss (EL)** requires:
$$\text{Expected Loss} = \text{PD} \times \text{LGD} \times \text{EAD}$$

Where:
- **PD**: Probability of Default
- **LGD**: Loss Given Default (% of exposure lost, e.g. 20% to 90%)
- **EAD**: Exposure at Default ($ amount owed at time of default)

Let's simulate a coupled macroeconomic environment where borrower characteristics simultaneously drive correlated movements in both LGD and EAD.

In [ ]:
n_samples = 15000

# Borrower and Loan Features
credit_score = np.random.normal(680, 50, n_samples)
debt_to_income = np.random.uniform(0.1, 0.6, n_samples)
collateral_value = np.random.uniform(50000, 500000, n_samples)
loan_amount = collateral_value * np.random.uniform(0.5, 0.95, n_samples)
interest_rate = 0.03 + (850 - credit_score) * 0.0002 + debt_to_income * 0.05
macro_unemployment = np.random.uniform(0.03, 0.12, n_samples)

X = pd.DataFrame({
    'credit_score': credit_score,
    'dti': debt_to_income,
    'collateral_value': collateral_value,
    'loan_amount': loan_amount,
    'interest_rate': interest_rate,
    'macro_unemployment': macro_unemployment
})

# Latent economic stress factor coupling both targets
stress_factor = 0.4 * debt_to_income + 0.3 * macro_unemployment - 0.0005 * (credit_score - 680)

# Target 1: LGD (0.0 to 1.0 fraction of loss)
lgd_true = np.clip(0.35 + 1.2 * stress_factor - 0.2 * (collateral_value / loan_amount) + np.random.normal(0, 0.05, n_samples), 0.05, 0.95)

# Target 2: EAD (Exposure dollar amount drawn at default)
ead_true = np.clip(loan_amount * (0.6 + 0.8 * stress_factor + np.random.normal(0, 0.04, n_samples)), loan_amount * 0.2, loan_amount * 1.05)

Y = np.column_stack([lgd_true, ead_true])

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

print(f"Training Feature Matrix: {X_train.shape}")
print(f"Training Multi-Target Matrix: {Y_train.shape} (Targets: LGD [%], EAD [$])")
print(f"Target Correlation (LGD vs. EAD): {np.corrcoef(Y_train[:, 0], Y_train[:, 1])[0, 1]:.3f}")

## 3. Native Multi-Target XGBoost vs. Independent Estimators

We now benchmark two architectures:
1. **Native Multi-Output XGBoost**: Pass `Y_train` directly (shape `(N, 2)`) to `xgb.XGBRegressor(tree_method='hist')`.
2. **Independent MultiOutputRegressor**: Fit 2 distinct single-target `XGBRegressor` models.

In [ ]:
# ---------------------------------------------------------
# 1. Native Multi-Target XGBoost (Vector-Leaf Tree)
# ---------------------------------------------------------
start_native = time.time()
native_model = xgb.XGBRegressor(
    tree_method='hist',
    n_estimators=100,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.8,
    random_state=42
)
native_model.fit(X_train, Y_train)
train_time_native = time.time() - start_native

# Inference benchmark
start_inf = time.time()
for _ in range(50):
    Y_pred_native = native_model.predict(X_test)
inf_time_native = (time.time() - start_inf) / 50 * 1000

# ---------------------------------------------------------
# 2. Independent Scikit-Learn Wrapper (2 Separate Trees)
# ---------------------------------------------------------
start_indep = time.time()
indep_base = xgb.XGBRegressor(
    tree_method='hist',
    n_estimators=100,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.8,
    random_state=42
)
indep_model = MultiOutputRegressor(indep_base)
indep_model.fit(X_train, Y_train)
train_time_indep = time.time() - start_indep

# Inference benchmark
start_inf = time.time()
for _ in range(50):
    Y_pred_indep = indep_model.predict(X_test)
inf_time_indep = (time.time() - start_inf) / 50 * 1000

print("=== BENCHMARK: NATIVE MULTI-OUTPUT vs INDEPENDENT ENSEMBLES ===")
print(f"Native Multi-Output Training Time: {train_time_native:.3f} s | Inference: {inf_time_native:.2f} ms")
print(f"Independent Multi-Output Training: {train_time_indep:.3f} s | Inference: {inf_time_indep:.2f} ms")
print(f"Training Speedup: {train_time_indep / train_time_native:.2f}x faster")

## 4. Performance & Residual Covariance Evaluation

Let's measure $R^2$ scores across both targets and verify how well the joint model captures the joint distribution.

In [ ]:
r2_lgd_native = r2_score(Y_test[:, 0], Y_pred_native[:, 0])
r2_ead_native = r2_score(Y_test[:, 1], Y_pred_native[:, 1])

r2_lgd_indep = r2_score(Y_test[:, 0], Y_pred_indep[:, 0])
r2_ead_indep = r2_score(Y_test[:, 1], Y_pred_indep[:, 1])

print(f"LGD R2 Score -> Native: {r2_lgd_native:.4f} | Independent: {r2_lgd_indep:.4f}")
print(f"EAD R2 Score -> Native: {r2_ead_native:.4f} | Independent: {r2_ead_indep:.4f}")

# Visualizing Joint Prediction
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(Y_test[:300, 0], Y_pred_native[:300, 0], alpha=0.6, color='teal', label='Native Vector-Leaf')
axes[0].plot([0, 1], [0, 1], '--k', alpha=0.7)
axes[0].set_title("Target 1: Loss Given Default (LGD)")
axes[0].set_xlabel("True LGD")
axes[0].set_ylabel("Predicted LGD")
axes[0].legend()

axes[1].scatter(Y_test[:300, 1] / 1e3, Y_pred_native[:300, 1] / 1e3, alpha=0.6, color='darkorange', label='Native Vector-Leaf')
min_val = min(Y_test[:300, 1].min(), Y_pred_native[:300, 1].min()) / 1e3
max_val = max(Y_test[:300, 1].max(), Y_pred_native[:300, 1].max()) / 1e3
axes[1].plot([min_val, max_val], [min_val, max_val], '--k', alpha=0.7)
axes[1].set_title("Target 2: Exposure at Default ($k EAD)")
axes[1].set_xlabel("True EAD ($k)")
axes[1].set_ylabel("Predicted EAD ($k)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Multi-Label Classification via Vector Softprob

For multi-label classification (where a single transaction or customer can trigger multiple non-exclusive flags, e.g., `AML_Watchlist`, `High_Velocity_Risk`, `Synthetic_Identity`), XGBoost can model multi-label binary classification natively.

In [ ]:
# Multi-label binary targets (3 correlated risk flags)
flag_aml = (stress_factor > np.percentile(stress_factor, 85)).astype(int)
flag_velocity = ((stress_factor + np.random.normal(0, 0.1, n_samples)) > np.percentile(stress_factor, 80)).astype(int)
flag_synthetic = ((stress_factor + np.random.normal(0, 0.15, n_samples)) > np.percentile(stress_factor, 90)).astype(int)

Y_multilabel = np.column_stack([flag_aml, flag_velocity, flag_synthetic])
X_train_ml, X_test_ml, Y_train_ml, Y_test_ml = train_test_split(X, Y_multilabel, test_size=0.2, random_state=42)

# Train multi-label classifier
ml_clf = xgb.XGBClassifier(
    tree_method='hist',
    n_estimators=80,
    max_depth=4,
    learning_rate=0.1,
    random_state=42
)
ml_clf.fit(X_train_ml, Y_train_ml)
ml_preds = ml_clf.predict_proba(X_test_ml)

print("=== MULTI-LABEL RISK FLAGS PROBABILITY PREDICTIONS (First 5 Samples) ===")
labels = ['AML_Watchlist_Prob', 'Velocity_Risk_Prob', 'Synthetic_Identity_Prob']
df_ml_results = pd.DataFrame(ml_preds[:5], columns=labels)
print(df_ml_results)